In [1]:
# !pip install ultralytics scikit-learn -q

In [2]:
import os
import glob
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from ultralytics import YOLO
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)
from PIL import Image

In [3]:
# Set path and parameters
data_dir = '../dataset_split'
results_dir = '../results'
os.makedirs(results_dir, exist_ok=True)

classes = ['biological', 'plastic']
img_size = 224
epochs = 30
batch_size = 16
model_name = 'yolo11n-cls.pt'

In [4]:
# Check dataset
for split in ['train', 'val', 'test']:
    for cls in classes:
        folder = os.path.join(data_dir, split, cls)
        count = len([
            f for f in os.listdir(folder)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])
        print(f'{split}/{cls}: {count} images')

train/biological: 729 images
train/plastic: 729 images
val/biological: 156 images
val/plastic: 156 images
test/biological: 157 images
test/plastic: 157 images


In [5]:
# Train YOLOv11
model = YOLO(model_name)

results = model.train(
    data = data_dir,
    epochs = epochs,
    imgsz = img_size,
    batch = batch_size,
    project = 'runs',
    name = 'yolov11_smartbin',
    patience = 5,
    verbose = True
)

New https://pypi.org/project/ultralytics/8.4.53 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.40  Python-3.13.5 torch-2.10.0+cpu CPU (13th Gen Intel Core i7-13620H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../dataset_split, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov11_

In [6]:
# Load best model
found = glob.glob('runs/**/best.pt', recursive=True) + glob.glob('../**/best.pt', recursive=True)

# Keep only yolov11 results
found = [f for f in found if 'yolov11' in f]

print('Found best.pt files:')
for f in found:
    print(f' -> {f}')

if found:
    best_model_path = max(found, key=os.path.getmtime)
    best_model = YOLO(best_model_path)
    print(f'Best model loaded from : {best_model_path}')
else:
    print('best.pt no found')

Found best.pt files:
 -> runs\classify\runs\yolov11_smartbin\weights\best.pt
 -> runs\classify\runs\yolov11_smartbin-2\weights\best.pt
 -> runs\classify\runs\yolov11_smartbin-5\weights\best.pt
 -> ..\notebooks\runs\classify\runs\yolov11_smartbin\weights\best.pt
 -> ..\notebooks\runs\classify\runs\yolov11_smartbin-2\weights\best.pt
 -> ..\notebooks\runs\classify\runs\yolov11_smartbin-5\weights\best.pt
Best model loaded from : runs\classify\runs\yolov11_smartbin-5\weights\best.pt


In [7]:
# Evaluate on test set
y_true = []
y_pred = []

for label_idx, cls in enumerate(classes):
    cls_dir = os.path.join(data_dir, 'test', cls)
    images = glob.glob(os.path.join(cls_dir, '*.jpg')) + glob.glob(os.path.join(cls_dir, '*.jpeg')) + glob.glob(os.path.join(cls_dir, '*.png'))

    print(f'Predicting {cls}: {len(images)} images')
    for img_path in images:
        result = best_model.predict(img_path, verbose=False)
        pred_idx = result[0].probs.top1
        y_true.append(label_idx)
        y_pred.append(pred_idx)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

Predicting biological: 157 images
Predicting plastic: 157 images


In [8]:
acc = accuracy_score(y_true, y_pred)
print(f'Test Accuracy: {acc*100:.2f}%')
print(f'Total images: {len(y_true)}')
print(f'Correct : {np.sum(y_true == y_pred)}')
print(f'Wrong : {np.sum(y_true != y_pred)}')

Test Accuracy: 96.50%
Total images: 314
Correct : 303
Wrong : 11


In [9]:
# Classification report 
print('CLASSIFICATION REPORT YOLOv11')
print('='*60)
print(classification_report(
    y_true,
    y_pred,
    target_names = classes
))


CLASSIFICATION REPORT YOLOv11
              precision    recall  f1-score   support

  biological       0.96      0.97      0.97       157
     plastic       0.97      0.96      0.96       157

    accuracy                           0.96       314
   macro avg       0.96      0.96      0.96       314
weighted avg       0.96      0.96      0.96       314



In [10]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot       = True,
    fmt         = 'd',
    cmap        = 'Oranges',
    xticklabels = classes,
    yticklabels = classes,
    linewidths  = 0.5
)
ax.set_title('Confusion Matrix — YOLOv11', fontsize=13, fontweight='bold')
ax.set_ylabel('Actual')
ax.set_xlabel('Predicted')

plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'yolov11_confusion_matrix.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved to results/yolov11_confusion_matrix.png')
print(f'\n  True Biological  (correct): {cm[0][0]}')
print(f'  False Plastic    (wrong)  : {cm[0][1]}')
print(f'  False Biological (wrong)  : {cm[1][0]}')
print(f'  True Plastic     (correct): {cm[1][1]}')

<Figure size 600x500 with 2 Axes>

Saved to results/yolov11_confusion_matrix.png

  True Biological  (correct): 152
  False Plastic    (wrong)  : 5
  False Biological (wrong)  : 6
  True Plastic     (correct): 151


In [11]:
# Sample predictions
all_paths = []
for cls in classes:
    cls_dir = os.path.join(data_dir, 'test', cls)
    for f in os.listdir(cls_dir):
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            all_paths.append((os.path.join(cls_dir, f), cls))

random.seed(42)
samples = random.sample(all_paths, min(8, len(all_paths)))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Sample Predictions — YOLOv11',
             fontsize=13, fontweight='bold')
axes = axes.flatten()

for i, (path, true_label) in enumerate(samples):
    result     = best_model.predict(path, verbose=False)
    pred_idx   = result[0].probs.top1
    pred_label = classes[pred_idx]
    confidence = result[0].probs.top1conf.item() * 100
    is_correct = pred_label == true_label

    img = Image.open(path).convert('RGB')
    axes[i].imshow(img)
    axes[i].axis('off')
    color  = '#4CAF50' if is_correct else '#F44336'
    status = '✓' if is_correct else '✗'
    axes[i].set_title(
        f'{status} Pred: {pred_label}\n{confidence:.1f}% | True: {true_label}',
        color=color, fontsize=8, fontweight='bold'
    )

plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'yolov11_sample_predictions.png'),
            dpi=150, bbox_inches='tight')
plt.show()

<Figure size 1600x800 with 8 Axes>

In [12]:
# Summary
precision = precision_score(y_true, y_pred)
recall    = recall_score(y_true, y_pred)
f1        = f1_score(y_true, y_pred)

print('='*50)
print('      EVALUATION SUMMARY — YOLOv11')
print('='*50)
print(f'  Test Accuracy : {acc*100:.2f}%')
print(f'  Precision     : {precision*100:.2f}%')
print(f'  Recall        : {recall*100:.2f}%')
print(f'  F1-Score      : {f1*100:.2f}%')
print(f'='*50)

      EVALUATION SUMMARY — YOLOv11
  Test Accuracy : 96.50%
  Precision     : 96.79%
  Recall        : 96.18%
  F1-Score      : 96.49%
